In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score,
                              recall_score, f1_score)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

BASE_PATH = r"../dataset"
CLASSES   = ['Rook', 'Queen', 'Pawn', 'Knight', 'King', 'Bishop']
IMG_SIZE  = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 30

print("Kutuphaneler yuklendi.")
print(f"TensorFlow: {tf.__version__}")

ModuleNotFoundError: No module named 'tqdm'

In [ ]:
X = []
y = []

print("Resimler yukleniyor...")

for label, class_name in enumerate(CLASSES):
    class_path = os.path.join(BASE_PATH, class_name)
    img_files  = os.listdir(class_path)

    for img_name in tqdm(img_files, desc=f"{class_name}"):
        img_path = os.path.join(class_path, img_name)
        try:
            img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
            X.append(np.array(img) / 255.0)   # Normalizasyon
            y.append(label)
        except:
            pass

X = np.array(X, dtype=np.float32)
y = np.array(y)

print(f"\nToplam resim : {len(X)}")
print(f"Veri sekli   : {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nEgitim seti : {len(X_train)} resim")
print(f"Test seti   : {len(X_test)}  resim")

In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator()  # Test icin augmentation yok

train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)
test_generator  = test_datagen.flow(X_test,  y_test,  batch_size=BATCH_SIZE, shuffle=False)

# Augmentation ornegini gorsellestir
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
aug_iter  = train_datagen.flow(X_train[0:1], batch_size=1)
for ax in axes:
    ax.imshow(next(aug_iter)[0])
    ax.axis('off')
plt.suptitle('Data Augmentation Ornekleri', fontsize=13)
plt.tight_layout()
plt.show()

print("Data augmentation hazir.")

In [ ]:
# Temel model: ImageNet agirliklariyla MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Asama 1: dondur

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax')  # 6 sinif
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_chess_model.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=3, min_lr=1e-6, verbose=1)
]

# --- ASAMA 1: Feature Extraction ---
print("Asama 1 basliyor: Feature Extraction...")
history1 = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)

# --- ASAMA 2: Fine-Tuning ---
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),   # 100x kucuk LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nAsama 2 basliyor: Fine-Tuning (son 30 katman acildi)...")
history2 = model.fit(
    train_generator,
    epochs=15,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)

# --- Egitim grafigi ---
def merge(h1, h2):
    merged = {}
    for k in h1.history:
        merged[k] = h1.history[k] + h2.history[k]
    return merged

hist  = merge(history1, history2)
split = len(history1.history['accuracy'])
ep    = range(1, len(hist['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(ep, hist['accuracy'],     label='Egitim', color='royalblue')
ax1.plot(ep, hist['val_accuracy'], label='Validation', color='tomato')
ax1.axvline(x=split, color='gray', linestyle='--', label='Fine-Tuning baslangici')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, hist['loss'],     label='Egitim', color='royalblue')
ax2.plot(ep, hist['val_loss'], label='Validation', color='tomato')
ax2.axvline(x=split, color='gray', linestyle='--')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Egitim Sureci', fontsize=14)
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()